# Step 2–3: Screening Eligibility, Test Assignment & Results

This notebook covers the screening layer of the simulation:
- **Step 2**: Determine which cancer screenings a patient is eligible for,
  assign the appropriate test modality, and handle unscreened patients.
- **Step 3**: Draw screening results (cervical: granular 6-category multinomial;
  other cancers: binary NEGATIVE/POSITIVE stubs).

All logic lives in `screening.py`. This notebook demonstrates and tests it.

**Handoff**: patients arrive here after being *seen* by a provider in
`01_arrivals.ipynb` (Sophia's layer). Results feed into `03_results_followup.ipynb`.

In [ ]:
import sys, random
sys.path.insert(0, '../src')   # ensure local .py files are importable

import config as cfg
from patient import Patient
from population import sample_patient
from screening import (
    get_eligible_screenings,
    is_due_for_screening,
    get_cervical_age_stratum,
    assign_screening_test,
    draw_cervical_result,
    draw_other_cancer_result,
    run_screening_step,
    handle_unscreened,
)

random.seed(cfg.RANDOM_SEED)
print('Imports OK')

## Eligibility Checks
Each cancer type has an eligibility function in `screening.py`.
`get_eligible_screenings(patient)` returns the full list for one patient.

In [7]:
# Edge-case patients to verify eligibility logic
cases = [
    # (label, age, has_cervix, smoker, pack_years, bmi)
    ('Age 25, with cervix, non-smoker', 25, True,  False, 0,  25),
    ('Age 45, with cervix, smoker 25pk', 45, True,  True,  25, 25),
    ('Age 45, no cervix',               45, False, False, 0,  25),
    ('Age 55, smoker 25pk',             55, True,  True,  25, 25),
    ('Age 68, BMI 17',                  68, True,  False, 0,  17),
    ('Age 70, all eligible',            70, True,  True,  25, 25),
    ('Age 20, below all thresholds',    20, True,  False, 0,  25),
]

print(f'{"Label":<35}  {"Eligible for"}' )
print('-' * 70)
for label, age, has_cervix, smoker, pack_years, bmi in cases:
    p = sample_patient(0, 0, 'pcp', 'outpatient')
    p.age, p.has_cervix, p.smoker = age, has_cervix, smoker
    p.pack_years, p.bmi = pack_years, bmi
    eligible = get_eligible_screenings(p)
    print(f'{label:<35}  {", ".join(eligible) if eligible else "(none)"}')

Label                                Eligible for
----------------------------------------------------------------------
Age 25, with cervix, non-smoker      cervical
Age 45, with cervix, smoker 25pk     cervical, breast, colorectal
Age 45, no cervix                    breast, colorectal
Age 55, smoker 25pk                  cervical, lung, breast, colorectal
Age 68, BMI 17                       breast, colorectal, osteoporosis
Age 70, all eligible                 lung, breast, colorectal, osteoporosis
Age 20, below all thresholds         (none)


## Test Assignment
Cervical test is age-stratified (ASCCP guidelines). Others use a single default.

In [8]:
ages = [22, 35, 50, 66]
print(f'{"Age":<6}  {"Stratum":<10}  {"Assigned test"}')
print('-' * 40)
for age in ages:
    p = sample_patient(0, 0, 'gynecologist', 'outpatient')
    p.age, p.has_cervix = age, True
    stratum = get_cervical_age_stratum(age)
    test    = assign_screening_test(p, 'cervical')
    print(f'{age:<6}  {stratum:<10}  {test}')

Age     Stratum     Assigned test
----------------------------------------
22      young       cytology
35      middle      co_test
50      middle      co_test
66      older       ineligible


## Result Distribution Check
Draw 5 000 results for a middle-aged patient and verify the distribution
matches the config probabilities (sanity check).

In [9]:
from collections import Counter

N = 5_000
p_template = sample_patient(0, 0, 'gynecologist', 'outpatient')
p_template.age, p_template.has_cervix = 42, True
p_template.hpv_positive, p_template.prior_cin = False, None

results = [draw_cervical_result(p_template, 'co_test') for _ in range(N)]
counts  = Counter(results)

print(f'Cervical result distribution (n={N:,}, age=42, co-test):')
for cat in ['NORMAL','ASCUS','LSIL','ASC-H','HSIL','HPV_POS_NORMAL_CYTO']:
    cnt = counts.get(cat, 0)
    expected = cfg.CERVICAL_RESULT_PROBS['middle_cytology'].get(cat, 0)
    print(f'  {cat:<30} observed={cnt/N*100:5.1f}%  expected={expected*100:5.1f}%')

Cervical result distribution (n=5,000, age=42, co-test):
  NORMAL                         observed= 90.5%  expected= 90.0%
  ASCUS                          observed=  3.2%  expected=  3.5%
  LSIL                           observed=  2.9%  expected=  3.0%
  ASC-H                          observed=  1.5%  expected=  1.5%
  HSIL                           observed=  0.9%  expected=  1.0%
  HPV_POS_NORMAL_CYTO            observed=  0.9%  expected=  1.0%


## Smoke Test — Run 30 Patients Through Screening
Simulates one provider-seen day: generate patients, run `run_screening_step`
for each eligible cancer, collect results.

In [10]:
from collections import defaultdict

random.seed(42)
DAY = 0
N_PATIENTS = 30

patients = [sample_patient(i, DAY, 'gynecologist', 'outpatient') for i in range(N_PATIENTS)]
screening_log = defaultdict(list)

for p in patients:
    eligible = get_eligible_screenings(p)
    if not eligible:
        outcome = handle_unscreened(p, DAY)
        screening_log['unscreened'].append(outcome)
        continue
    for cancer in eligible:
        result = run_screening_step(p, cancer, DAY)
        if result:
            screening_log[cancer].append(result)

print(f'Results across {N_PATIENTS} patients:\n')
for cancer, results in sorted(screening_log.items()):
    cnt = Counter(results)
    print(f'  {cancer}:')
    for r, c in sorted(cnt.items()):
        print(f'    {r:<30} {c}')

# Show one patient trace
print('\n── Sample patient trace ──')
patients[0].print_history()

Results across 30 patients:

  breast:
    NEGATIVE                       16
    POSITIVE                       1
  cervical:
    ASC-H                          1
    HPV_POS_NORMAL_CYTO            5
    LSIL                           1
    NORMAL                         19
  colorectal:
    NEGATIVE                       13
    POSITIVE                       1
  lung:
    NEGATIVE                       1
  osteoporosis:
    NEGATIVE                       4

── Sample patient trace ──

── Patient 0 | age=50 | destination=gynecologist ──
  Day     0: SCREEN breast via mammogram
  Day     0: RESULT breast: POSITIVE
  Day     0: SCREEN colorectal via colonoscopy
  Day     0: RESULT colorectal: NEGATIVE
